# unsloth/Qwen2-0.5B-Instruct

---



In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [17]:
from datasets import load_from_disk

dataset= load_from_disk("/content/drive/MyDrive/qwen_funetune/final/dataset_items")

len(dataset)

35

In [18]:
dataset[0]

{'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'},
  {'content': "What is the price of the item called 'Polka Style 30x4.4x12.7 cm'?",
   'role': 'user'},
  {'content': '$599', 'role': 'assistant'}],
 'text': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 20 Sep 2025\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the price of the item called 'Polka Style 30x4.4x12.7 cm'?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n$599<|eot_id|>"}

In [11]:
for i in range(35):
    print(dataset[i]['messages'])


[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': "What is the price of the item called 'Polka Style 30x4.4x12.7 cm'?", 'role': 'user'}, {'content': '$599', 'role': 'assistant'}]
[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'Is there a guarantee on this product?', 'role': 'user'}, {'content': 'Yes, it comes with a three-year warranty.', 'role': 'assistant'}]
[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the price of the Kryuchok Line collection?', 'role': 'user'}, {'content': '$699', 'role': 'assistant'}]
[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'How many pieces are included in the collection?', 'role': 'user'}, {'content': '2', 'role': 'assistant'}]
[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the size of a runo nica tumbay in cm?', 'role': 'user'}, {'content': 'A runo nica tumbay has dimensions of 30 x 20 x 20 cm.',

In [12]:
from huggingface_hub import login
login('.')  # авторизация

### Fine-tuning Synthetic Dataset with Unsloth

In [13]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2-0.5B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 09-20 07:08:01 [__init__.py:244] Automatically detected platform cuda.
ERROR 09-20 07:08:03 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.7: Fast Qwen2 patching. Transformers: 4.55.4. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Будем обучать метод LoRA



In [14]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.9.7 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


<a name="Data"></a>
### Data Prep

ChatML for **unsloth/Qwen2-0.5B-Instruct**
```
<|im_start|>system
You are a helpful assistant.
<|im_end|>

<|im_start|>user
What is the price of the item called 'Polka Style 30x4.4x12.7 cm'?
<|im_end|>

<|im_start|>assistant
$599
<|im_end|>

```

In [19]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Get our previous dataset and format it:
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [20]:
dataset[0]

{'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'},
  {'content': "What is the price of the item called 'Polka Style 30x4.4x12.7 cm'?",
   'role': 'user'},
  {'content': '$599', 'role': 'assistant'}],
 'text': "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is the price of the item called 'Polka Style 30x4.4x12.7 cm'?<|im_end|>\n<|im_start|>assistant\n$599<|im_end|>\n"}

<a name="Train"></a>
### Train the model


In [21]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        max_steps = 200,
        num_train_epochs=1,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/35 [00:00<?, ? examples/s]

In [22]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
0.555 GB of memory reserved.


In [23]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 35 | Num Epochs = 40 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
1,2.921600
2,3.104400
3,2.852400
4,2.382900
5,2.107100
6,2.308400
7,2.145900
8,1.435400
9,1.425300
10,2.049600


In [25]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

321.2644 seconds used for training.
5.35 minutes used for training.
Peak reserved memory = 0.877 GB.
Peak reserved memory for training = 0.322 GB.
Peak reserved memory % of max memory = 5.949 %.
Peak reserved memory for training % of max memory = 2.184 %.


<a name="Inference"></a>
### Inference


In [37]:
concrete_questions = [
    "How much does the 'Polka Style 30x4.4x12.7 cm' item cost?",
    "What is the price for items in the Kryuchok Line collection?",
    "What are the dimensions of a runo nica tumbay in centimeters?",
    "What is the weight of a runo nica tumbay in kilograms?",
    "Does Runo offer any customizations for their furniture?",
    "How much does the Stakan Dlya Zubnykh Shchyetok Nastennyy Line cost?",
    "What is the price of runo nicica's tumbay i komody tumba prikrovatnaya nid?"
]


In [35]:
#  мои данные
output_file = "/content/drive/MyDrive/qwen_funetune/json/parsed_data.txt"

with open(output_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

blocks = []
current_block = []

for line in lines:
    line = line.strip()
    if line == "========================================":
        if current_block:
            blocks.append(current_block)
            current_block = []
    elif line:
        current_block.append(line)

if current_block:
    blocks.append(current_block)

for item in blocks[0]:
    print(item)

id: 80602754
name: Полка BERKRAFT Style 30x4.4x12.7 см
category: 2271
url: https://hoff.ru/catalog/tovary_dlya_doma/aksessuary_dlya_vannoy/hranenie_v_vannoy/navesnye_polki/polka_style_id10050709/?articul=80602754
price: 599
description:
BERKRAFT: Бренд
Style: Коллекция
Китай: Производитель
24 мес.: Гарантия
Год: 2: Срок службы
30x4,4x12,7 см: Размер (ШхВхГ)
алюминий: Материал
чёрный: Цвет
0,327 кг: Вес
3 кг: Максимальная нагрузка общая
Да: С крючками
Нет: Метод крепления
клейкая лента: Метод крепления
2024-01-17: addedDate


# Cмотрим результаты файнтьюна:

In [36]:
for question in concrete_questions:
    messages = [
        {"role": "user", "content": question},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    output_ids = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7
    )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "assistant" in decoded:
        answer = decoded.split("assistant")[-1].strip()
    else:
        answer = decoded.strip()

    print(f"Вопрос: {question}")
    print(f"Ответ: {answer}")
    print("\n" + "="*40 + "\n")


Вопрос: How much does the 'Polka Style 30x4.4x12.7 cm' item cost?
Ответ: $599


Вопрос: What is the price for items in the Kryuchok Line collection?
Ответ: $699


Вопрос: What are the dimensions of a runo nica tumbay in centimeters?
Ответ: The dimensions of a runo nica tumbay are 30 x 20 x 20 cm.


Вопрос: What is the weight of a runo nica tumbay in kilograms?
Ответ: The weight of a runo nica tumbay is 30999 grams.


Вопрос: Does Runo offer any customizations for their furniture?
Ответ: Yes, they offer customizations such as attaching a rug to the back or changing the color of the wood.


Вопрос: Do you know the size of the Stakan Dlya Zubnykh Shchyetok Nastennyy Line?
Ответ: The size of the Stakan Dlya Zubnykh Shchyetok Nastennyy Line is 2024-05-14: addedDate.


Вопрос: How much does the Stakan Dlya Zubnykh Shchyetok Nastennyy Line cost?
Ответ: $999


Вопрос: What is the price of runo nicica's tumbay i komody tumba prikrovatnaya nid?
Ответ: The price of runo nicica's tumbay i komody t

<a name="Save"></a>


In [38]:
SAVE_NAME = 'Qwen2-items_funetuned'
SAVE_PATH='/content/drive/MyDrive/qwen_funetune'

model.save_pretrained(f'{SAVE_PATH}/{SAVE_NAME}')
tokenizer.save_pretrained(f'{SAVE_PATH}/{SAVE_NAME}')


('/content/drive/MyDrive/qwen_funetune/Qwen2-items_funetuned/tokenizer_config.json',
 '/content/drive/MyDrive/qwen_funetune/Qwen2-items_funetuned/special_tokens_map.json',
 '/content/drive/MyDrive/qwen_funetune/Qwen2-items_funetuned/chat_template.jinja',
 '/content/drive/MyDrive/qwen_funetune/Qwen2-items_funetuned/vocab.json',
 '/content/drive/MyDrive/qwen_funetune/Qwen2-items_funetuned/merges.txt',
 '/content/drive/MyDrive/qwen_funetune/Qwen2-items_funetuned/added_tokens.json',
 '/content/drive/MyDrive/qwen_funetune/Qwen2-items_funetuned/tokenizer.json')

# Посмотрим что бы ответила модель до дообучения:

In [41]:
torch.cuda.empty_cache()
del model

In [42]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2-0.5B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

==((====))==  Unsloth 2025.9.7: Fast Qwen2 patching. Transformers: 4.55.4. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [43]:
for question in concrete_questions:
    messages = [
        {"role": "user", "content": question},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    output_ids = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7
    )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "assistant" in decoded:
        answer = decoded.split("assistant")[-1].strip()
    else:
        answer = decoded.strip()

    print(f"Вопрос: {question}")
    print(f"Ответ сырой модели: {answer}")
    print("\n" + "="*40 + "\n")


Вопрос: How much does the 'Polka Style 30x4.4x12.7 cm' item cost?
Ответ сырой модели: The price of Polka Style 30x4.4x12.7 cm can vary depending on several factors, including the specific item and its location or manufacturer. However, in general, these items typically cost between $15 to $20 for a single unit.

It's best to check directly with the manufacturer or retailer for the most accurate pricing information. Additionally, you may want to consider other options available online that might be more affordable. It's always a good idea to compare prices from different sources before making a purchase.


Вопрос: What is the price for items in the Kryuchok Line collection?
Ответ сырой модели: I'm sorry, but as an AI language model, I do not have access to current pricing information or specific product prices. You may want to check with a retailer or online store that carries the Kryuchok Line collection to obtain accurate and up-to-date information on the price of products.


Вопрос: 

# **ИТОГИ**:

### 1) Подготовка данных для дообучения LoRA

Использовала https://github.com/meta-llama/synthetic-data-kit.

1. Подготовка данных о товарах.
2. Разбиение данных на части для поэтапной генерации QA-пар.
3. Генерация вопросов и ответов для товаров.
4. Преобразование QA-пар в формат для дообучения.
5. Формирование датасета сообщений: system → user → assistant.

**Пример формата данных:**
```json
{
  "messages": [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the price of the item called 'Polka Style 30x4.4x12.7 cm'?"},
    {"role": "assistant", "content": "$599"}
  ]
}


### 2) Дообучение LoRA на Qwen2-0.5B-Instruct

- Ранг матриц (r) = 16  
- Всего 1 эпоха


### 3)  Результаты работы модели после LoRA

После дообучения модель корректно отвечает на вопросы о товарах, точно указывает цены, размеры, вес и возможности кастомизации, в отличие от исходной модели, которая давала общие, неточные ответы и часто добавляла лишние пояснения.  
